In [41]:
import pandas as pd
DATASET_PATH = "../GAIA-DataSet-main/run/run_table_2021-07.csv"
df = pd.read_csv(DATASET_PATH)
# df = df[df.datetime==""]

In [34]:
df.head()

,datetime,service,message
0,2021-07-01,dbservice1,"2021-07-01 11:54:27,616 | WARNING | 0.0.0.4 | ..."
1,2021-07-01,dbservice1,"2021-07-01 12:25:08,804 | WARNING | 0.0.0.4 | ..."
2,2021-07-01,dbservice1,"2021-07-01 22:33:05,033 | WARNING | 0.0.0.4 | ..."
3,2021-07-01,dbservice2,"2021-07-01 12:54:56,480 | WARNING | 0.0.0.2 | ..."
4,2021-07-01,dbservice2,"2021-07-01 15:10:23,517 | WARNING | 0.0.0.2 | ..."


In [7]:
df = df[df.datetime=="2021-07-20"]


In [4]:
df

,datetime,service,message
9333,2021-07-20,dbservice1,"2021-07-20 03:10:06,148 | ERROR | 0.0.0.4 | 17..."
9334,2021-07-20,dbservice1,(Background on this error at: http://sqlalche....
9335,2021-07-20,dbservice1,"2021-07-20 03:11:41,056 | ERROR | 0.0.0.4 | 17..."
9336,2021-07-20,dbservice1,(Background on this error at: http://sqlalche....
9337,2021-07-20,dbservice1,"2021-07-20 03:11:41,088 | ERROR | 0.0.0.4 | 17..."
...,...,...,...
9881,2021-07-20,logservice2,"2021-07-20 03:54:29,635 | ERROR | 0.0.0.2 | 17..."
9882,2021-07-20,logservice2,(Background on this error at: http://sqlalche....
9883,2021-07-20,logservice2,"2021-07-20 03:54:29,666 | ERROR | 0.0.0.2 | 17..."
9884,2021-07-20,logservice2,(Background on this error at: http://sqlalche....


In [42]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import json
import re
from datetime import datetime

def parse_log_line(line: str) -> dict:
    """
    解析单行日志，返回结构化字典。
    支持三种格式：
      1) 标准6字段：timestamp | level | src_ip | svc_ip | service | message
      2) 标准7字段：timestamp | level | src_ip | svc_ip | service | track_id | message
      3) 无结构文本（如 SQLAlchemy 错误提示）
    """
    line = line.rstrip('\n')

    # 尝试按 " | " 分割（注意空格）
    parts = line.split(" | ")
    # 常见的无结构错误消息（第三种格式）
    if len(parts) not in (6, 7):
        # 如果是纯文本（如以 "(Background on this error" 开头）
        return {
            "level": "NONE",          # 默认级别，可调整
            "message": line
        }

    # 有结构日志：至少6个部分
    timestamp_str = parts[0].strip()
    level = parts[1].strip()
    src_ip = parts[2].strip()
    svc_ip = parts[3].strip()
    service = parts[4].strip()

    # 区分6字段还是7字段
    if len(parts) == 6:
        track_id = None
        message = parts[5].strip()
    else:  # len == 7
        track_id = parts[5].strip()
        message = parts[6].strip()

    # 转换时间戳（可选，保留原始字符串和解析后的ISO格式）
    parsed_ts = None
    try:
        # 格式：2021-07-01 15:10:23,517 -> 替换逗号为小数点
        normalized = timestamp_str.replace(',', '.')
        dt = datetime.strptime(normalized, "%Y-%m-%d %H:%M:%S.%f")
        parsed_ts = dt.isoformat()
    except ValueError:
        parsed_ts = None

    result = {
        "timestamp_raw": timestamp_str,
        "timestamp_iso": parsed_ts,
        "level": level,
        "src_ip": src_ip,
        "service_ip": svc_ip,
        "service": service,
        "message": message,
    }
    if track_id:
        result["track_id"] = track_id

    return result


In [43]:
split_result = df['message'].apply(lambda x: parse_log_line(x))

In [25]:
pd.DataFrame(split_result.values.tolist())

,type,timestamp_raw,timestamp_iso,level,src_ip,service_ip,service,message,track_id,raw
0,structured,"2021-07-01 11:54:27,616",2021-07-01T11:54:27.616000,WARNING,0.0.0.4,172.17.0.3,dbservice1,[memory_anomalies] trigger a high memory progr...,NaN,NaN
1,structured,"2021-07-01 12:25:08,804",2021-07-01T12:25:08.804000,WARNING,0.0.0.4,172.17.0.3,dbservice1,[normal memory freed label] lasts ten minutes,NaN,NaN
2,structured,"2021-07-01 22:33:05,033",2021-07-01T22:33:05.033000,WARNING,0.0.0.4,172.17.0.3,dbservice1,[memory_anomalies] trigger a high memory progr...,NaN,NaN
3,structured,"2021-07-01 12:54:56,480",2021-07-01T12:54:56.480000,WARNING,0.0.0.2,172.17.0.2,dbservice2,[memory_anomalies] trigger a high memory progr...,NaN,NaN
4,structured,"2021-07-01 15:10:23,517",2021-07-01T15:10:23.517000,WARNING,0.0.0.2,172.17.0.2,dbservice2,[memory_anomalies] trigger a high memory progr...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
17150,structured,"2021-07-31 03:51:53,965",2021-07-31T03:51:53.965000,INFO,0.0.0.2,172.17.0.3,logservice2,upload business logs on 2021-07-30 successfully,NaN,NaN
17151,structured,"2021-07-31 03:58:55,311",2021-07-31T03:58:55.311000,INFO,0.0.0.2,172.17.0.3,logservice2,upload trace logs on 2021-07-30 successfully,NaN,NaN
17152,structured,"2021-07-31 03:58:55,428",2021-07-31T03:58:55.428000,INFO,0.0.0.2,172.17.0.3,logservice2,upload run_logs logs on 2021-07-30 successfully,NaN,NaN
17153,structured,"2021-07-31 13:17:00,460",2021-07-31T13:17:00.460000,WARNING,0.0.0.2,172.17.0.3,logservice2,[memory_anomalies] trigger a high memory progr...,NaN,NaN


In [44]:
df = pd.concat([df, pd.DataFrame(split_result.values.tolist())], axis=1)

In [28]:
df.shape

(17155, 14)

In [45]:
df.to_excel("结构化数据2.xlsx")